In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel('ADITI_CLASS_ALL_DATA.xlsx')
df.head()

,imo,vessel,report_date_time,report_type,status,sea_state,cargo_total,cargo_total_teu,speed_by_log,speed_by_gps,...,derived_total_fo_type,first_total_fo,actual_total_fo,distance,speed,mean_draft,derived_speed,actual_speed,derived_status,activity_time
0,9235581,MSC ADITI,2017-07-09 23:54:00,SAIL,IN PORT,0,4442.0,329,0.0,0.00,...,0.0,2.2,2.2,0.00,0.00,6.7,NaN,0.00,IN PORT,0.0
1,9235581,MSC ADITI,2017-07-10 01:00:00,COSP,DRIFTING,3,4442.0,329,0.0,0.00,...,0.0,1.3,1.3,8.00,0.00,6.7,7.272727,7.27,SEA-DRIFT,1.1
2,9235581,MSC ADITI,2017-07-10 12:00:00,NOON,AT SEA,3,4442.0,329,17.5,18.00,...,0.0,26.5,26.5,198.00,18.00,6.7,18.000000,18.00,SEA-DRIFT,11.0
3,9235581,MSC ADITI,2017-07-10 23:00:00,EOSP,AT SEA,3,4442.0,329,17.1,17.27,...,0.0,25.8,25.8,189.97,17.27,6.7,17.270000,17.27,SEA-DRIFT,11.0
4,9235581,MSC ADITI,2017-07-11 02:12:00,BRTH,DRIFTING,0,4442.0,329,0.0,0.00,...,0.0,4.0,4.0,24.00,0.00,6.7,7.500000,7.50,SEA-DRIFT,3.2


In [3]:
aditi_data = df[(df['imo'] == 9235581)  ][[  
    
        'report_date_time', 
        'derived_status',
        'actual_speed', 
        'mean_draft',  
        'sea_state',  
        'me_actual_steaming_time',  

        'ae_t_steaming',
        'aux_running', 
        'blr_running', 

        'distance',  
        'actual_total_fo',
        'total_co_2',  

        'date_time',
        'date',
        'year', 
        'adjusted_date',  
        'activity_time'
     
     
    ]]
aditi_data.head(2)

,report_date_time,derived_status,actual_speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,distance,actual_total_fo,total_co_2,date_time,date,year,adjusted_date,activity_time
0,2017-07-09 23:54:00,IN PORT,0.00,6.7,0,0.0,7.8,2,0,0.0,2.2,7.0532,2017-07-09 23:54:00,2017-07-09,2017,2017-07-09,0.0
1,2017-07-10 01:00:00,SEA-DRIFT,7.27,6.7,3,1.1,2.2,2,0,8.0,1.3,4.0482,2017-07-10 01:00:00,2017-07-10,2017,2017-07-09,1.1


In [6]:
# Count the number of 'IN PORT' rows
in_port_count = (aditi_data['derived_status'] == 'IN PORT').sum()

# Count the number of 'NOT IN PORT' rows
not_in_port_count = (aditi_data['derived_status'] != 'IN PORT').sum()

print("IN PORT count:", in_port_count)
print("NOT IN PORT count:", not_in_port_count)


IN PORT count: 1410
NOT IN PORT count: 3907


In [8]:
# aditi_data[[ 'date_time' ,  'me_actual_steaming_time', 'activity_time', 'derived_status']].to_excel('Activity_time.xlsx', index = False)

# original code 

# New code:

In [11]:
def find_clean_duplicate_report_time(df):
    print("df.shape", df.shape)
    new = []
    dupli_dates = []
    for i in df['report_date_time']:
        if i not in new:
            new.append(i)
        else:
            dupli_dates.append(i)
            
    port_reports = aditi_data[aditi_data['report_date_time'].isin(dupli_dates) &  (aditi_data['derived_status'] == 'IN PORT')
                             &  (aditi_data['me_actual_steaming_time'] == 0)
                             ]
    print("port_reports.shape",port_reports.shape)
    
    # Skipping day creation (groupby and aggregation)
    port_df = port_reports  # Keep the port_reports as-is, no aggregation
    
    print(port_df.shape)
    
    sail_reports = aditi_data[aditi_data['report_date_time'].isin(dupli_dates) &  (aditi_data['derived_status'] != 'IN PORT')
                             & (aditi_data['actual_speed'] > 0) & (aditi_data['me_actual_steaming_time'] > 0) & (aditi_data['aux_running'] > 0)
                             ]
    
    print("sail_reports.shape",sail_reports.shape)
    
    # Skipping day creation (groupby and aggregation)
    sail_df = sail_reports  # Keep the sail_reports as-is, no aggregation
    
    print("sail_df.shape",sail_df.shape)
    print(df[~df['report_date_time'].isin(dupli_dates)].shape)
    
    global finaldf
    finaldf = pd.concat([df[~df['report_date_time'].isin(dupli_dates)], port_df, sail_df], axis = 0).reset_index(drop = True)
    print("finaldf.shape",finaldf.shape)
    
    return finaldf 

find_clean_duplicate_report_time(aditi_data)


df.shape (5317, 17)
port_reports.shape (5, 17)
(5, 17)
sail_reports.shape (43, 17)
sail_df.shape (43, 17)
(5239, 17)
finaldf.shape (5287, 17)


,report_date_time,derived_status,actual_speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,distance,actual_total_fo,total_co_2,date_time,date,year,adjusted_date,activity_time
0,2017-07-09 23:54:00,IN PORT,0.00,6.70,0,0.0,7.8,2,0,0.00,2.20,7.05320,2017-07-09 23:54:00,2017-07-09,2017,2017-07-09,0.0
1,2017-07-10 01:00:00,SEA-DRIFT,7.27,6.70,3,1.1,2.2,2,0,8.00,1.30,4.04820,2017-07-10 01:00:00,2017-07-10,2017,2017-07-09,1.1
2,2017-07-10 12:00:00,SEA-DRIFT,18.00,6.70,3,11.0,11.0,1,0,198.00,26.50,82.52100,2017-07-10 12:00:00,2017-07-10,2017,2017-07-09,11.0
3,2017-07-10 23:00:00,SEA-DRIFT,17.27,6.70,3,11.0,0.0,0,0,189.97,25.80,80.34120,2017-07-10 23:00:00,2017-07-10,2017,2017-07-10,11.0
4,2017-07-11 02:12:00,SEA-DRIFT,7.50,6.70,0,3.2,6.4,2,0,24.00,4.00,12.45600,2017-07-11 02:12:00,2017-07-11,2017,2017-07-10,3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282,2024-04-03 00:00:00,SEA-DRIFT,8.93,10.25,2,2.8,6.6,3,1,25.00,2.75,8.56350,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,24.0
5283,2024-04-03 00:00:00,SEA-DRIFT,7.74,10.25,2,2.7,7.1,3,1,20.90,2.25,7.00650,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,0.0
5284,2024-04-03 00:00:00,SEA-DRIFT,9.60,10.25,4,2.5,3.8,2,1,24.00,1.90,5.91660,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,0.0
5285,2024-04-03 00:00:00,SEA-DRIFT,9.20,10.25,2,1.0,1.8,2,1,9.00,0.61,1.89954,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,0.0


In [13]:
# Count the number of 'IN PORT' rows
in_port_count = (finaldf['derived_status'] == 'IN PORT').sum()

# Count the number of 'NOT IN PORT' rows
not_in_port_count = (finaldf['derived_status'] != 'IN PORT').sum()

print("IN PORT count:", in_port_count)
print("NOT IN PORT count:", not_in_port_count)


IN PORT count: 1409
NOT IN PORT count: 3878


In [15]:
finaldf

,report_date_time,derived_status,actual_speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,distance,actual_total_fo,total_co_2,date_time,date,year,adjusted_date,activity_time
0,2017-07-09 23:54:00,IN PORT,0.00,6.70,0,0.0,7.8,2,0,0.00,2.20,7.05320,2017-07-09 23:54:00,2017-07-09,2017,2017-07-09,0.0
1,2017-07-10 01:00:00,SEA-DRIFT,7.27,6.70,3,1.1,2.2,2,0,8.00,1.30,4.04820,2017-07-10 01:00:00,2017-07-10,2017,2017-07-09,1.1
2,2017-07-10 12:00:00,SEA-DRIFT,18.00,6.70,3,11.0,11.0,1,0,198.00,26.50,82.52100,2017-07-10 12:00:00,2017-07-10,2017,2017-07-09,11.0
3,2017-07-10 23:00:00,SEA-DRIFT,17.27,6.70,3,11.0,0.0,0,0,189.97,25.80,80.34120,2017-07-10 23:00:00,2017-07-10,2017,2017-07-10,11.0
4,2017-07-11 02:12:00,SEA-DRIFT,7.50,6.70,0,3.2,6.4,2,0,24.00,4.00,12.45600,2017-07-11 02:12:00,2017-07-11,2017,2017-07-10,3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282,2024-04-03 00:00:00,SEA-DRIFT,8.93,10.25,2,2.8,6.6,3,1,25.00,2.75,8.56350,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,24.0
5283,2024-04-03 00:00:00,SEA-DRIFT,7.74,10.25,2,2.7,7.1,3,1,20.90,2.25,7.00650,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,0.0
5284,2024-04-03 00:00:00,SEA-DRIFT,9.60,10.25,4,2.5,3.8,2,1,24.00,1.90,5.91660,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,0.0
5285,2024-04-03 00:00:00,SEA-DRIFT,9.20,10.25,2,1.0,1.8,2,1,9.00,0.61,1.89954,2024-04-03 00:00:00,2024-04-03,2024,2024-04-02,0.0


In [17]:
finaldf['derived_status'].value_counts()

derived_status
SEA-DRIFT    3878
IN PORT      1409
Name: count, dtype: int64

In [19]:
finaldf.isnull().sum()

report_date_time           0
derived_status             0
actual_speed               0
mean_draft                 0
sea_state                  0
me_actual_steaming_time    0
ae_t_steaming              0
aux_running                0
blr_running                0
distance                   0
actual_total_fo            0
total_co_2                 0
date_time                  0
date                       0
year                       0
adjusted_date              0
activity_time              0
dtype: int64

# original code


# new code:

In [51]:
def prepare_day_data(df):
    print("Initial Data Shape:", df.shape)

    # Filter for Port data where 'derived_status' is 'IN PORT'
    global dfport
    dfport = df[df['derived_status'] == 'IN PORT'][['adjusted_date', 'actual_speed', 'mean_draft', 'sea_state',
                                                    'me_actual_steaming_time', 'ae_t_steaming', 'aux_running',
                                                    'blr_running', 'distance', 'actual_total_fo', 'total_co_2',
                                                    'activity_time']]

    # Set steaming time to 0 for port data
    dfport['me_actual_steaming_time'] = 0
    dfport.rename(columns={'actual_speed': 'speed'}, inplace=True)

    # Separate Port data where speed, distance, and steaming time are all 0
    port_df = dfport[(dfport['speed'] == 0) & (dfport['distance'] == 0) & (dfport['me_actual_steaming_time'] == 0)]

    # Filter for Sailing data where 'derived_status' is not 'IN PORT'
    global dfsail
    dfsail = df[df['derived_status'] != 'IN PORT'][['adjusted_date', 'actual_speed', 'mean_draft', 'sea_state',
                                                    'me_actual_steaming_time', 'ae_t_steaming', 'aux_running',
                                                    'blr_running', 'distance', 'actual_total_fo', 'total_co_2',
                                                    'activity_time']]

    # Rename column for consistency
    dfsail.rename(columns={'actual_speed': 'speed'}, inplace=True)

    # Filter Sailing data where speed, distance, and steaming time are all greater than 0
    sail_df = dfsail[(dfsail['speed'] > 0) & (dfsail['distance'] > 0) & (dfsail['me_actual_steaming_time'] > 0)]

    # Remove records with steaming time > 26 hours
    sail_df = sail_df[sail_df['me_actual_steaming_time'] <= 26]
    
    print("Port Data Shape:", dfport.shape)
    print("Filtered Sail Data Shape:", sail_df.shape)

    # Separate Port data where speed, distance, and steaming time are 0
    port_day_df = dfport[(dfport['speed'] == 0) & (dfport['distance'] == 0) & (dfport['me_actual_steaming_time'] == 0)]

    # Concatenate Sail and Port dataframes
    final_sail_df = sail_df.round(2).drop_duplicates().reset_index(drop=True)
    final_sail_df['status'] = 'SAIL'

    # Further filter Sailing data for valid speed and draft
    final_sail_df = final_sail_df[(final_sail_df['speed'] <= 20) & (final_sail_df['speed'] > 0)
                                  & (final_sail_df['mean_draft'] > 0) & (final_sail_df['me_actual_steaming_time'] > 0)]

    final_port_df = port_df.round(2).drop_duplicates().reset_index(drop=True)
    final_port_df['status'] = 'PORT'

    print("Final Sail Data Shape:", final_sail_df.shape)
    print("Final Port Data Shape:", final_port_df.shape)

    # Concatenate final Sail and Port data
    global all_df
    all_df = pd.concat([final_sail_df, final_port_df]).round(2).drop_duplicates().reset_index(drop=True)

    # Round auxiliary and boiler running times
    all_df['aux_running'] = all_df['aux_running'].apply(lambda x: round(x, 0))
    all_df['blr_running'] = all_df['blr_running'].apply(lambda x: round(x, 0))

    return all_df


In [53]:
prepare_day_data(finaldf)


Initial Data Shape: (5287, 17)
Port Data Shape: (1409, 12)
Filtered Sail Data Shape: (3702, 12)
Final Sail Data Shape: (3660, 13)
Final Port Data Shape: (1409, 13)


,adjusted_date,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,distance,actual_total_fo,total_co_2,activity_time,status
0,2017-07-09,7.27,6.70,3,1.1,2.2,2,0,8.00,1.30,4.05,1.1,SAIL
1,2017-07-09,18.00,6.70,3,11.0,11.0,1,0,198.00,26.50,82.52,11.0,SAIL
2,2017-07-10,17.27,6.70,3,11.0,0.0,0,0,189.97,25.80,80.34,11.0,SAIL
3,2017-07-10,7.50,6.70,0,3.2,6.4,2,0,24.00,4.00,12.46,3.2,SAIL
4,2017-07-11,9.41,7.60,3,1.7,3.4,2,0,16.00,3.20,9.96,1.7,SAIL
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5064,2019-08-30,0.00,6.65,3,0.0,11.0,2,0,0.00,2.22,7.12,0.0,PORT
5065,2024-03-30,0.00,10.60,2,0.0,13.2,2,1,0.00,3.69,11.49,1.5,PORT
5066,2024-03-31,0.00,11.40,4,0.0,8.4,3,1,0.00,2.00,6.23,0.0,PORT
5067,2024-04-01,0.00,10.25,4,0.0,14.1,3,1,0.00,3.35,10.43,24.0,PORT


In [49]:
all_df

,adjusted_date,speed,mean_draft,sea_state,me_actual_steaming_time,ae_t_steaming,aux_running,blr_running,distance,actual_total_fo,total_co_2,activity_time,status
0,2017-07-09,7.27,6.70,3,1.1,2.2,2,0,8.00,1.30,4.05,1.1,SAIL
1,2017-07-09,18.00,6.70,3,11.0,11.0,1,0,198.00,26.50,82.52,11.0,SAIL
2,2017-07-10,17.27,6.70,3,11.0,0.0,0,0,189.97,25.80,80.34,11.0,SAIL
3,2017-07-10,7.50,6.70,0,3.2,6.4,2,0,24.00,4.00,12.46,3.2,SAIL
4,2017-07-11,9.41,7.60,3,1.7,3.4,2,0,16.00,3.20,9.96,1.7,SAIL
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5064,2019-08-30,0.00,6.65,3,0.0,11.0,2,0,0.00,2.22,7.12,0.0,PORT
5065,2024-03-30,0.00,10.60,2,0.0,13.2,2,1,0.00,3.69,11.49,1.5,PORT
5066,2024-03-31,0.00,11.40,4,0.0,8.4,3,1,0.00,2.00,6.23,0.0,PORT
5067,2024-04-01,0.00,10.25,4,0.0,14.1,3,1,0.00,3.35,10.43,24.0,PORT


In [59]:
all_df.to_excel("MSC_ADITI_CLEANED_DATA_corrected.xlsx", index = False)

In [55]:
all_df['status'].value_counts()

status
SAIL    3660
PORT    1409
Name: count, dtype: int64

In [57]:
all_df.isnull().sum()

adjusted_date              0
speed                      0
mean_draft                 0
sea_state                  0
me_actual_steaming_time    0
ae_t_steaming              0
aux_running                0
blr_running                0
distance                   0
actual_total_fo            0
total_co_2                 0
activity_time              0
status                     0
dtype: int64